# Complex Tasks Demo — Multi-Shape Layouts with LLM Reasoning

Two progressively harder draw.io tasks, each shown **three ways**:

1. **Manual (deterministic)** — we call `dispatch()` step by step to prove the task is achievable.
2. **LLM-driven** — we hand a structured natural-language goal to the Executor and watch it sequence the same operands.
3. **Saved compound tool** — successful traces are crystallised with `save_trace_as_tool`.

### The two tasks

| # | Task | New concepts |
|---|------|--------------|
| 1 | **Counterclockwise ring** — 4 rectangles (node1–node4) arranged in a 2×2 grid, edges forming a CCW cycle starting at top-left | Multi-shape positioning; directed cycle |
| 2 | **Server-client star** — 3 clients above 1 server, all edges labelled "connection" | Star topology; `label_edge` operand |

### Prerequisites
- draw.io running and accessible by pyautogui.
- macOS Screen Recording permission for this terminal / Jupyter kernel.
- `state/ui_graph.json` calibrated for your draw.io window.

In [ ]:
# --- 0. Setup ---------------------------------------------------------------
import os, sys, time, json
sys.path.insert(0, os.path.abspath('..'))

import pyautogui
from core import config
from core.tools import dispatch, TOOL_CATALOG
from core.tools.save_tool import save_trace_as_tool, check_trace_success
from core.state import scene_graph as sg
from core.agents.executor import infer
from core.capture import screenshot as _screenshot

print(f'Loaded {len(TOOL_CATALOG)} tools.')
print(f'label_edge available: {"label_edge" in TOOL_CATALOG}')

# Shared helpers
def show_scene(g):
    """Print the current scene graph summary."""
    print(sg.summary_for_prompt(g['scene_graph']))

def clean_canvas(g, n_undos=50):
    """5-second countdown, then undo loop + reset scene graph."""
    sg.reset()
    g['scene_graph'] = sg.load()
    g['selected_handles'] = None
    print('Switch to draw.io NOW.')
    for i in range(5, 0, -1):
        print(f'  {i}s …')
        time.sleep(1)
    print('GO — cleaning canvas …')
    pyautogui.click(1100, 800); time.sleep(0.3)
    for _ in range(n_undos):
        pyautogui.hotkey('command', 'z'); time.sleep(0.04)
    pyautogui.click(1100, 800); time.sleep(0.3)
    print('clean. scene_graph reset.')

def run_llm_loop(task: str, g: dict, max_steps: int = 35):
    """Single-tool-per-turn LLM loop. Returns (trace, history)."""
    history = []
    trace   = []

    for step in range(1, max_steps + 1):
        print(f'\n{'='*50}')
        print(f'LLM step {step}/{max_steps}')
        print('='*50)

        img      = _screenshot(f'_complex_step{step}.png')
        # screenshot+SG mode — infer() attaches the image because
        # screenshot_path is not None.  See core/agents/executor.py.
        decision = infer(task, g, screenshot_path=img,
                         history=history or None)
        tool     = decision.get('tool')
        params   = decision.get('params', {}) or {}
        reasoning = decision.get('reasoning', '')

        print(f'reasoning : {reasoning}')
        print(f'tool      : {tool}({params})')

        if tool == 'task_complete':
            print('\n✅ LLM signalled task_complete')
            break

        if tool == 'request_rescan':
            dispatch('scan_handles', {}, ui_graph=g)
            history.append({'role': 'assistant',
                            'content': json.dumps(decision)})
            history.append({'role': 'user',
                            'content': 'rescanned — continue'})
            continue

        if tool not in TOOL_CATALOG:
            print(f'❌ unknown tool "{tool}" — aborting')
            break

        result = dispatch(tool, params, ui_graph=g)
        status = result.get('status')
        print(f'result    : status={status}'
              + (f', error={result.get("error")}' if status != 'ok' else ''))

        trace.append({'tool': tool, 'params': dict(params), 'result': result})
        history.append({'role': 'assistant',
                        'content': json.dumps({'tool': tool,
                                               'params': params,
                                               'reasoning': reasoning})})
        history.append({'role': 'user',
                        'content': (
                            f"Tool '{tool}' executed (status={status}). "
                            "What is the next step? "
                            "Use task_complete if ALL objectives are met."
                        )})

        print('\n--- scene graph after this step ---')
        show_scene(g)
        time.sleep(0.3)

    else:
        print(f'\n⚠  reached max_steps={max_steps} without task_complete')

    return trace, history

# Shared ui_graph (calibrated coords + scene_graph threaded through all calls)
G = config.ui_graph()
G['scene_graph'] = sg.load()
print('\nSetup complete.')

---
## Task 1 — Counterclockwise Ring of 4 Nodes

Place four rectangles in a 2×2 grid, then connect them with directed edges forming a **counterclockwise cycle** starting at top-left:

```
  node1 ←─────── node2
    │                 ↑
    ↓                 │
  node4 ──────→ node3
```

Target edge set:
- `node1 → node4`  (down the left side,   source_anchor=s)
- `node4 → node3`  (right along the bottom, source_anchor=e)
- `node3 → node2`  (up the right side,    source_anchor=n)
- `node2 → node1`  (left along the top,   source_anchor=w)

### Layout plan (canvas coords)

| Shape | Target center | Moves from default drop zone (~751, 534) |
|-------|--------------|-------------------------------------------|
| node1 | (591, 414)   | west 160 + north 120 |
| node2 | (911, 414)   | east 160 + north 120 |
| node3 | (911, 634)   | east 160 + south 100 |
| node4 | (591, 634)   | west 160 + south 100 |

In [ ]:
# --- T1 · Manual · Step 0: focus + clean ---------------------------------
clean_canvas(G)

In [ ]:
# --- T1 · Manual · Step 1: place and position all four nodes -------------
#
# draw.io always drops new shapes at the same default canvas position.
# We immediately move each shape to its grid slot (two move_shape calls,
# one per axis) before placing the next one — otherwise bboxes overlap.

SHAPES = [
    ('node1', 'w', 160, 'n', 120),
    ('node2', 'e', 160, 'n', 120),
    ('node3', 'e', 160, 's', 100),
    ('node4', 'w', 160, 's', 100),
]

for label, dir1, amt1, dir2, amt2 in SHAPES:
    print(f'\n── placing {label} ──')
    dispatch('place_shape',  {'tool_name': 'Rectangle_Tool'}, ui_graph=G)
    dispatch('type_label',   {'text': label},                 ui_graph=G)
    dispatch('press_escape', {},                              ui_graph=G)
    print(f'  moving: {dir1} {amt1}px then {dir2} {amt2}px')
    dispatch('move_shape',   {'direction': dir1, 'amount': amt1}, ui_graph=G)
    dispatch('move_shape',   {'direction': dir2, 'amount': amt2}, ui_graph=G)
    dispatch('click_empty_canvas', {},                        ui_graph=G)
    time.sleep(0.2)

print('\n--- scene graph after placing all nodes ---')
show_scene(G)

In [ ]:
# --- T1 · Manual · Step 2: connect counterclockwise ----------------------
#
# source_anchor chosen so the edge leaves the shape on the side that
# faces the target (avoids backtrack routing artefacts in draw.io).

CCW_EDGES = [
    ('node1', 'node4', 's'),   # top-left → bottom-left  (down)
    ('node4', 'node3', 'e'),   # bottom-left → bottom-right (right)
    ('node3', 'node2', 'n'),   # bottom-right → top-right (up)
    ('node2', 'node1', 'w'),   # top-right → top-left (left)
]

for src, tgt, anchor in CCW_EDGES:
    print(f'  connecting {src} →[{anchor}]→ {tgt}')
    r = dispatch('connect_shapes',
                 {'source_id': src, 'target_id': tgt,
                  'source_anchor': anchor},
                 ui_graph=G)
    print(f'    status={r.get("status")}'
          + (f'  error={r.get("error")}' if r.get('status') != 'ok' else ''))
    time.sleep(0.4)

print('\n--- final scene graph (Task 1 manual) ---')
show_scene(G)

# Structural validation
scene = G['scene_graph']
labels = {o['label'] for o in scene['objects']}
n_edges = len(scene['edges'])
print(f'\nObjects: {len(scene["objects"])}  Labels: {sorted(labels)}')
print(f'Edges:   {n_edges}')
if labels == {'node1','node2','node3','node4'} and n_edges == 4:
    print('\n🎉 Task 1 (manual) complete — 4 nodes + 4 CCW edges.')
else:
    print('\n⚠  unexpected scene graph — inspect above')

### Task 1 — LLM run

The prompt below is deliberately **prescriptive**: it gives the LLM an exact step-by-step plan
so we can observe whether it follows structured instructions across ~24 tool calls.
The scene graph printed after each step makes the LLM's reasoning transparent.

**Why so detailed?**  For novel multi-step layouts, the LLM needs the spatial intent
spelled out — "top-left", "bottom-right", "counterclockwise" — because the scene graph
summary gives coordinates but not visual orientation.  The step-by-step plan anchors
the LLM to the correct sequencing even after the model has made several prior calls.

In [ ]:
# --- T1 · LLM · reset + clean --------------------------------------------
clean_canvas(G)

In [ ]:
# --- T1 · LLM · execute ---------------------------------------------------
#
# PROMPT DESIGN NOTES
# ─────────────────────────────────────────────────────────────────────────
# 1. Layout block: tells the LLM the INTENDED positions in human terms
#    (top-left, top-right …).  The scene graph gives numbers; this gives
#    intent — both together let the model self-correct if a move overshoots.
#
# 2. Move amounts: derived from the default drop position (~751,534 center)
#    and the target grid positions.  We give specific amounts so the LLM
#    doesn't have to invent them.
#
# 3. Connection table: explicit source_anchor values prevent the LLM from
#    choosing 'auto', which could pick the wrong side when shapes are close.
#
# 4. "One tool per turn" reminder: the framework loops back after each
#    dispatch, so the LLM must NOT batch multiple tools in one response.

TASK_1 = """\
GOAL: Place four labelled rectangles in a 2x2 grid on the canvas, then connect
them with directed edges forming a counterclockwise cycle starting at node1.

LAYOUT (target positions):
  node1 = top-left      node2 = top-right
  node4 = bottom-left   node3 = bottom-right

COUNTERCLOCKWISE EDGES:
  node1 → node4   (down the left side,     source_anchor='s')
  node4 → node3   (right along the bottom, source_anchor='e')
  node3 → node2   (up the right side,      source_anchor='n')
  node2 → node1   (left along the top,     source_anchor='w')

STEP-BY-STEP PLAN (execute exactly ONE tool per turn):
 1.  place_shape(Rectangle_Tool)
 2.  type_label('node1')
 3.  press_escape
 4.  move_shape('w', 160)            ← send node1 west to left column
 5.  move_shape('n', 120)            ← send node1 north to top row
 6.  place_shape(Rectangle_Tool)
 7.  type_label('node2')
 8.  press_escape
 9.  move_shape('e', 160)            ← send node2 east to right column
 10. move_shape('n', 120)            ← send node2 north to top row
 11. place_shape(Rectangle_Tool)
 12. type_label('node3')
 13. press_escape
 14. move_shape('e', 160)            ← send node3 east to right column
 15. move_shape('s', 100)            ← send node3 south to bottom row
 16. place_shape(Rectangle_Tool)
 17. type_label('node4')
 18. press_escape
 19. move_shape('w', 160)            ← send node4 west to left column
 20. move_shape('s', 100)            ← send node4 south to bottom row
 21. connect_shapes(source_id='node1', target_id='node4', source_anchor='s')
 22. connect_shapes(source_id='node4', target_id='node3', source_anchor='e')
 23. connect_shapes(source_id='node3', target_id='node2', source_anchor='n')
 24. connect_shapes(source_id='node2', target_id='node1', source_anchor='w')
 25. task_complete

RULES:
- Execute exactly one tool per turn.
- After every move_shape, read the updated SCENE GRAPH to confirm the bbox moved
  before continuing.
- If the scene graph shows an unexpected state (wrong bbox, missing object),
  call click_empty_canvas and re-read the scene graph before continuing.
- Emit task_complete only when the SCENE GRAPH shows 4 objects (node1–node4)
  and 4 edges forming the CCW cycle above.
"""

trace1, history1 = run_llm_loop(TASK_1, G, max_steps=35)

print('\n=========== FINAL scene graph (Task 1 LLM) ===========')
show_scene(G)
print(f'\n📦 Collected {len(trace1)} trace steps.')

In [ ]:
# --- T1 · Save trace as a compound tool ----------------------------------
results1 = [s['result'] for s in trace1]
ok1 = check_trace_success(results1)
print(f'All steps ok? {ok1}  ({len(trace1)} steps)')

if ok1:
    steps1 = [{'tool': s['tool'], 'params': s['params']} for s in trace1]
    defn1 = save_trace_as_tool(
        name='ccw_ring_4nodes',
        description=(
            'Place node1–node4 in a 2×2 grid and connect them counterclockwise: '
            'node1→node4→node3→node2→node1.  Learned from LLM trace.'
        ),
        params=[],
        steps=steps1,
        overwrite=True,
    )
    print('\nSaved as state/tools/ccw_ring_4nodes.json')
    print(f'Tool level: L{TOOL_CATALOG["ccw_ring_4nodes"].level}')
else:
    failed = [(i, s['tool'], s['result'].get('status'))
              for i, s in enumerate(trace1)
              if s['result'].get('status') != 'ok']
    print(f'Failed steps: {failed}')

---
## Task 2 — Server-Client Star Topology

Place a server at the bottom and three clients above it, connect each client
to the server, and label every edge **"connection"**.

```
  client1    client2    client3
     │           │           │
     └──────┬────┘           │
            │   connection   │
            └───────────────→server
```

This task introduces **`label_edge`** — a new L1 operand that:
1. Computes the midpoint of an edge from the stored anchor points.
2. Double-clicks there to open draw.io's inline edge-label editor.
3. Types the text and presses Escape.
4. Updates `edge.label` in the scene graph.

### Layout plan

| Shape   | Target center | Moves from default (~751, 534) |
|---------|--------------|--------------------------------|
| server  | (751, 634)   | south 100 |
| client1 | (551, 414)   | west 200 + north 120 |
| client2 | (751, 414)   | north 120 |
| client3 | (951, 414)   | east 200 + north 120 |

In [ ]:
# --- T2 · Manual · Step 0: focus + clean ---------------------------------
clean_canvas(G)

In [2]:
# --- T2 · Manual · Step 1: place server + clients ------------------------

T2_SHAPES = [
    # (label, [(dir, amount), ...])
    ('server',  [('s', 100)]),
    ('client1', [('w', 200), ('n', 120)]),
    ('client2', [('n', 120)]),
    ('client3', [('e', 200), ('n', 120)]),
]

for label, moves in T2_SHAPES:
    print(f'\n── placing {label} ──')
    dispatch('place_shape',  {'tool_name': 'Rectangle_Tool'}, ui_graph=G)
    dispatch('type_label',   {'text': label},                 ui_graph=G)
    dispatch('press_escape', {},                              ui_graph=G)
    for direction, amount in moves:
        print(f'  move_shape({direction}, {amount})')
        dispatch('move_shape', {'direction': direction, 'amount': amount}, ui_graph=G)
    dispatch('click_empty_canvas', {}, ui_graph=G)
    time.sleep(0.2)

print('\n--- scene graph after placing all shapes ---')
show_scene(G)

INFO   domains.drawio.operations:   [L1] place_shape('Rectangle_Tool') → click (36, 322) + Enter



── placing server ──


INFO   domains.drawio.operations:   [L1] type_label('server')
INFO   domains.drawio.operations:   [L1] press_escape
INFO   core.capture: Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
INFO   core.capture: Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png


  move_shape(s, 100)


INFO   domains.drawio.operations:   [L1] move_shape('s', 100) → escape+reclick, drag (750,536) → (750,636)
INFO   core.capture: Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
INFO   core.capture: Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png
INFO   core.tools.actions:   [L1] click_empty_canvas → (600, 400)
INFO   domains.drawio.operations:   [L1] place_shape('Rectangle_Tool') → click (36, 322) + Enter


...

  move_shape(e, 200)


INFO   domains.drawio.operations:   [L1] move_shape('e', 200) → escape+reclick, drag (750,536) → (950,536)
INFO   core.capture: Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
INFO   core.capture: Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png


  move_shape(n, 120)


INFO   domains.drawio.operations:   [L1] move_shape('n', 120) → escape+reclick, drag (953,536) → (953,416)
INFO   core.capture: Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
INFO   core.capture: Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png
INFO   core.tools.actions:   [L1] click_empty_canvas → (600, 400)



--- scene graph after placing all shapes ---
**Objects (10):**
  - `obj_001` Rectangle "server"  bbox=[447,247,67x46]
  - `obj_002` Rectangle "client1"  bbox=[505,396,84x42]
  - `obj_003` Rectangle "client2"  bbox=[704,396,84x42]
  - `obj_004` Rectangle "client3"  bbox=[911,396,84x42]
  - `obj_005` Rectangle "server"  bbox=[331,244,70x40]
  - `obj_006` Rectangle "client1"  bbox=[691,504,120x60]
  - `obj_007` Rectangle "server"  bbox=[690,578,112x112]
  - `obj_008` Rectangle "client1"  bbox=[505,396,84x42]
  - `obj_009` Rectangle "client2"  bbox=[704,396,84x42]
  - `obj_010` Rectangle "client3"  bbox=[911,396,84x42]

_(scene_graph op #42, last op: move_shape:n)_


In [ ]:
# --- T2 · Manual · Step 2: connect each client to server ----------------

for client in ('client1', 'client2', 'client3'):
    print(f'  connecting {client} → server')
    r = dispatch('connect_shapes',
                 {'source_id': client, 'target_id': 'server',
                  'source_anchor': 'auto'},
                 ui_graph=G)
    print(f'    status={r.get("status")}'
          + (f'  error={r.get("error")}' if r.get('status') != 'ok' else ''))
    time.sleep(0.4)

print('\n--- scene graph after connecting ---')
show_scene(G)

In [ ]:
# --- T2 · Manual · Step 3: label every edge "connection" ----------------
#
# label_edge(edge_id, text) computes the midpoint of the edge's path from
# the stored anchor points, double-clicks there, types the text, and
# updates edge.label in the scene graph.

scene = G['scene_graph']
print(f'Edges to label: {[e["id"] for e in scene["edges"]]}')

for edge in scene['edges']:
    eid = edge['id']
    print(f'\n  label_edge("{eid}", "connection")')
    r = dispatch('label_edge', {'edge_id': eid, 'text': 'connection'}, ui_graph=G)
    print(f'    status={r.get("status")}  at={r.get("at")}'
          + (f'  error={r.get("error")}' if r.get('status') != 'ok' else ''))
    time.sleep(0.4)

print('\n--- final scene graph (Task 2 manual) ---')
show_scene(G)

# Structural validation
edge_labels = {e['label'] for e in scene['edges']}
obj_labels  = {o['label'] for o in scene['objects']}
print(f'\nObjects: {sorted(obj_labels)}')
print(f'Edge labels: {edge_labels}')
if obj_labels == {'server','client1','client2','client3'} \
        and len(scene['edges']) == 3 \
        and edge_labels == {'connection'}:
    print('\n🎉 Task 2 (manual) complete — star topology with "connection" labels.')
else:
    print('\n⚠  unexpected scene graph — inspect above')

### Task 2 — LLM run

The prompt walks the LLM through a more complex sequence: placing one server +
three clients (each requiring two positioning moves), three connections,
and three edge-label calls.

**Key difference from Task 1**: the LLM must use `label_edge(edge_id, text)` which
requires reading the edge IDs from the SCENE GRAPH after each `connect_shapes` call.
The prompt explicitly instructs it to inspect the scene graph for the correct IDs
rather than guessing.

Expected trace length: ~28 steps.

In [ ]:
# --- T2 · LLM · reset + clean --------------------------------------------
clean_canvas(G)

In [ ]:
# --- T2 · LLM · execute ---------------------------------------------------
#
# PROMPT DESIGN NOTES
# ─────────────────────────────────────────────────────────────────────────
# 1. "after every connect_shapes, read the SCENE GRAPH EDGES block" —
#    tells the LLM to ground its label_edge calls in the actual IDs that
#    the framework assigned, not assumed sequential IDs.
#
# 2. label_edge description in the plan — the LLM has not seen this tool
#    used before in its conversation history, so we explain the semantics
#    (double-click midpoint, enter text) so it can reason about failure.
#
# 3. source_anchor='auto' — let the framework pick the best edge
#    attachment point; the exact direction depends on relative positions
#    the LLM cannot pre-compute from the prompt alone.

TASK_2 = """\
GOAL: Create a server-client star topology:
  - 1 server rectangle (label: 'server') at the BOTTOM CENTER of the canvas
  - 3 client rectangles (labels: 'client1', 'client2', 'client3') in a row
    ABOVE the server: client1 on the left, client2 in the center, client3 on the right
  - 3 directed edges: client1→server, client2→server, client3→server
  - Every edge must be labelled 'connection' (use label_edge)

STEP-BY-STEP PLAN (execute exactly ONE tool per turn):

 — Place server —
 1.  place_shape(Rectangle_Tool)
 2.  type_label('server')
 3.  press_escape
 4.  move_shape('s', 100)            ← move server below the default drop zone

 — Place client1 (top-left) —
 5.  place_shape(Rectangle_Tool)
 6.  type_label('client1')
 7.  press_escape
 8.  move_shape('w', 200)            ← client1 to left column
 9.  move_shape('n', 120)            ← client1 to top row

 — Place client2 (top-center) —
 10. place_shape(Rectangle_Tool)
 11. type_label('client2')
 12. press_escape
 13. move_shape('n', 120)            ← client2 to top row (already centered)

 — Place client3 (top-right) —
 14. place_shape(Rectangle_Tool)
 15. type_label('client3')
 16. press_escape
 17. move_shape('e', 200)            ← client3 to right column
 18. move_shape('n', 120)            ← client3 to top row

 — Connect all clients to server —
 19. connect_shapes(source_id='client1', target_id='server', source_anchor='auto')
 20. connect_shapes(source_id='client2', target_id='server', source_anchor='auto')
 21. connect_shapes(source_id='client3', target_id='server', source_anchor='auto')

 — Label every edge 'connection' —
 IMPORTANT: after the connect_shapes calls, read the SCENE GRAPH EDGES block
 to find the actual edge IDs assigned (edge_001, edge_002, edge_003).
 Then call label_edge for each one:
 22. label_edge(edge_id='edge_001', text='connection')
 23. label_edge(edge_id='edge_002', text='connection')
 24. label_edge(edge_id='edge_003', text='connection')

 25. task_complete

RULES:
- One tool per turn — never emit multiple tools in one response.
- After each connect_shapes, inspect the SCENE GRAPH EDGES block to confirm
  the edge was added before calling the next tool.
- For label_edge: the framework double-clicks the edge midpoint and types
  the text — you only need to supply the edge_id and text.
- Verify SCENE GRAPH has 4 objects + 3 edges with label='connection' before
  emitting task_complete.
"""

trace2, history2 = run_llm_loop(TASK_2, G, max_steps=35)

print('\n=========== FINAL scene graph (Task 2 LLM) ===========')
show_scene(G)
print(f'\n📦 Collected {len(trace2)} trace steps.')

In [ ]:
# --- T2 · Save trace as a compound tool ----------------------------------
results2 = [s['result'] for s in trace2]
ok2 = check_trace_success(results2)
print(f'All steps ok? {ok2}  ({len(trace2)} steps)')

if ok2:
    steps2 = [{'tool': s['tool'], 'params': s['params']} for s in trace2]
    defn2 = save_trace_as_tool(
        name='server_client_star',
        description=(
            'Place server (bottom) + client1/2/3 (top row), connect each '
            'client to server, label every edge "connection".  '
            'Learned from LLM trace.'
        ),
        params=[],
        steps=steps2,
        overwrite=True,
    )
    print('\nSaved as state/tools/server_client_star.json')
    print(f'Tool level: L{TOOL_CATALOG["server_client_star"].level}')
else:
    failed = [(i, s['tool'], s['result'].get('status'))
              for i, s in enumerate(trace2)
              if s['result'].get('status') != 'ok']
    print(f'Failed steps: {failed}')

---
## What these demos show

### Prompt structure matters for multi-step tasks

Both prompts use the same pattern:
1. **GOAL** — the visual outcome in human terms.
2. **LAYOUT** — intent-level position description ("top-left", "bottom center").
3. **STEP-BY-STEP PLAN** — every tool call with params, numbered.
4. **RULES** — invariants the LLM must preserve (one tool per turn, verify scene graph).

Without the step-by-step plan the LLM tends to:
- Forget to `press_escape` before moving, leaving a shape in text-edit mode.
- Place the second shape without moving the first, creating overlapping bboxes.
- Call `connect_shapes` before all bboxes are known, getting a "bbox missing" error.

### Scene graph as ground truth

The LLM sees the scene graph after every step, not just the screenshot.  
This lets it:
- Confirm a `move_shape` actually changed the bbox (not just assume it did).
- Read the exact `edge_NNN` IDs assigned by `connect_shapes` before calling `label_edge`.
- Self-correct: if `connect_shapes` returned ok but the edge is missing from the graph,
  the LLM can call `click_empty_canvas` and retry.

### label_edge — the new L1 operand

`label_edge(edge_id, text)` demonstrates the extension pattern:
- Python implementation in `domains/drawio/operations.py` (≈50 lines).
- JSON registration in `state/tools/label_edge.json` (8 lines).
- No changes to any other file — the loader picks it up automatically.
- The executor's TOOLS BY GOAL section was updated with one line so the LLM knows when to use it.

### Tool-learning flywheel

```
LLM explores (this notebook)
        ↓
Trace verified with check_trace_success()
        ↓
Persisted with save_trace_as_tool()   →  state/tools/ccw_ring_4nodes.json
        ↓                               state/tools/server_client_star.json
Available to future LLM sessions as one-call compound tools (L2+)
```